# Gradient Descent vs Newton's Method: Seeing the Difference

**Companion notebook for [Why Gradient Descent Works (And When It Doesn't)](https://codechitti216.github.io)**

In the blog post, we showed that gradient descent computes `.grad` using partial derivatives — which assume all other weights are constant. But they're not. Newton's method fixes this by using the Hessian to account for cross-parameter interactions.

Now let's see the difference in practice. We'll train a simple linear regression model both ways and compare:
- How many steps each method takes to converge
- The loss at every step
- What the Hessian actually looks like

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes

np.random.seed(42)

## Step 1: The Data

We'll use sklearn's diabetes dataset — 442 patients, 10 features, target is a measure of disease progression.

To keep things small and visual, we pick **just one feature** (BMI). Our model will learn two parameters: a slope and an intercept. That gives us a 2×2 Hessian we can actually look at.

In [ ]:
data = load_diabetes()
X_raw = data.data[:, 2].reshape(-1, 1)  # BMI feature (already standardized)
y = data.target

# Add a bias column (column of 1s for the intercept)
X = np.hstack([np.ones((X_raw.shape[0], 1)), X_raw])

print(f"X shape: {X.shape}")   # (442, 2) — 442 samples, 2 params (bias + BMI)
print(f"y shape: {y.shape}")   # (442,)
print(f"\nFirst 5 rows of X:")
print(X[:5])
print(f"\nFirst 5 targets: {y[:5]}")

Let's see what we're fitting — a scatter plot of BMI vs disease progression.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X_raw, y, alpha=0.5, s=20)
plt.xlabel('BMI (standardized)')
plt.ylabel('Disease Progression')
plt.title('Diabetes Dataset — BMI vs Disease Progression')
plt.grid(True, alpha=0.3)
plt.show()

## Step 2: The Loss Function

We're doing linear regression with Mean Squared Error. Our model predicts $\hat{y} = Xw$, and we want to minimize:

$$L(\mathbf{w}) = \frac{1}{n}\|X\mathbf{w} - \mathbf{y}\|^2$$

where $\mathbf{w} = [w_0, w_1]^T$ — the intercept and slope.

In [ ]:
def mse_loss(X, y, w):
    """Mean Squared Error loss."""
    residuals = X @ w - y
    return np.mean(residuals ** 2)

# Starting point: both weights at zero
w_init = np.zeros(2)
print(f"Initial loss: {mse_loss(X, y, w_init):.2f}")

## Step 3: Gradient Descent

The gradient of MSE with respect to $\mathbf{w}$ is:

$$\nabla L = \frac{2}{n}X^T(X\mathbf{w} - \mathbf{y})$$

This gradient is built from partial derivatives — each component computed by holding the other weight constant. We then update **both weights simultaneously** using this gradient.

This is exactly what `loss.backward()` computes. Each `.grad` is one component of this vector.

In [ ]:
def gradient_descent(X, y, lr, n_steps):
    """Train linear regression with gradient descent, tracking everything."""
    n = X.shape[0]
    w = np.zeros(X.shape[1])
    history = {'loss': [], 'weights': [], 'gradients': []}
    
    for step in range(n_steps):
        # Current loss
        residuals = X @ w - y
        loss = np.mean(residuals ** 2)
        history['loss'].append(loss)
        history['weights'].append(w.copy())
        
        # Gradient — partial derivatives assuming other weight is constant
        grad = (2 / n) * X.T @ residuals
        history['gradients'].append(grad.copy())
        
        # Update both weights simultaneously
        w = w - lr * grad
    
    # Record final state
    history['loss'].append(mse_loss(X, y, w))
    history['weights'].append(w.copy())
    
    return w, history

Before we run gradient descent, we need to pick a learning rate. Too large and we'll see the backfire we proved in the blog post. Let's first figure out what "too large" means for this specific problem.

The maximum stable learning rate for gradient descent on MSE is $\eta < \frac{2}{\lambda_{\max}}$, where $\lambda_{\max}$ is the largest eigenvalue of the Hessian $H = \frac{2}{n}X^TX$.

In [ ]:
n = X.shape[0]
H = (2 / n) * X.T @ X
eigenvalues = np.linalg.eigvalsh(H)

print(f"Hessian:\n{H}")
print(f"\nEigenvalues: {eigenvalues}")
print(f"Max stable learning rate: {2 / eigenvalues.max():.4f}")
print(f"\nWe'll use lr = 0.5 (safe) and also try lr = 1.5 (unstable) to see the backfire.")

### Gradient Descent with a Safe Learning Rate

In [ ]:
w_gd, hist_gd = gradient_descent(X, y, lr=0.5, n_steps=50)

print(f"Final weights: intercept = {w_gd[0]:.4f}, slope = {w_gd[1]:.4f}")
print(f"Final loss:    {hist_gd['loss'][-1]:.4f}")
print(f"\nLoss at each step:")
for i in range(min(10, len(hist_gd['loss']))):
    print(f"  Step {i:3d}: loss = {hist_gd['loss'][i]:.4f}")
print(f"  ...")
print(f"  Step {len(hist_gd['loss'])-1:3d}: loss = {hist_gd['loss'][-1]:.4f}")

### Gradient Descent with a Dangerously Large Learning Rate

Now let's see the backfire. If we use a learning rate above the stability threshold, the simultaneous weight updates will interfere destructively — just like our $z = x^2 + xy + y^2$ example with $\eta = 0.8$.

In [ ]:
w_gd_unstable, hist_gd_unstable = gradient_descent(X, y, lr=1.5, n_steps=20)

print(f"Loss at each step (lr=1.5):")
for i in range(min(10, len(hist_gd_unstable['loss']))):
    print(f"  Step {i:3d}: loss = {hist_gd_unstable['loss'][i]:.2f}")
print(f"\nThe loss explodes. That's the backfire.")

## Step 4: Newton's Method

Now let's do it the "correct" way. Newton's method uses the Hessian to account for cross-parameter interactions:

$$\Delta \mathbf{w} = -H^{-1} \nabla L$$

where $H = \frac{2}{n}X^TX$ is the Hessian.

For linear regression, the Hessian is **constant** — it doesn't change as the weights change. This means Newton's method should find the exact minimum in a **single step**. That single step is equivalent to the Normal Equation:

$$\mathbf{w}^* = (X^TX)^{-1}X^T\mathbf{y}$$

In [ ]:
def newtons_method(X, y, n_steps=5):
    """Train linear regression with Newton's method, tracking everything."""
    n = X.shape[0]
    w = np.zeros(X.shape[1])
    history = {'loss': [], 'weights': [], 'gradients': []}
    
    # Hessian is constant for linear regression
    H = (2 / n) * X.T @ X
    H_inv = np.linalg.inv(H)
    
    for step in range(n_steps):
        # Current loss
        residuals = X @ w - y
        loss = np.mean(residuals ** 2)
        history['loss'].append(loss)
        history['weights'].append(w.copy())
        
        # Gradient
        grad = (2 / n) * X.T @ residuals
        history['gradients'].append(grad.copy())
        
        # Newton's update — Hessian-adjusted gradient
        w = w - H_inv @ grad
    
    # Record final state
    history['loss'].append(mse_loss(X, y, w))
    history['weights'].append(w.copy())
    
    return w, history

In [ ]:
w_newton, hist_newton = newtons_method(X, y, n_steps=5)

print(f"Final weights: intercept = {w_newton[0]:.4f}, slope = {w_newton[1]:.4f}")
print(f"\nLoss at each step:")
for i, loss in enumerate(hist_newton['loss']):
    print(f"  Step {i}: loss = {loss:.4f}")

One step. The loss drops to the minimum on the very first update and stays there.

Let's verify this matches the Normal Equation (closed-form solution):

In [ ]:
# Normal Equation: w* = (X^T X)^{-1} X^T y
w_exact = np.linalg.inv(X.T @ X) @ X.T @ y

print(f"Newton's method weights:  {w_newton}")
print(f"Normal Equation weights:  {w_exact}")
print(f"Difference:               {np.abs(w_newton - w_exact).max():.2e}")
print(f"\nThey're the same (up to floating point). Newton's method IS the Normal Equation for linear regression.")

## Step 5: The Comparison

Let's put it all together. How does the loss evolve for gradient descent vs Newton's method?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Loss curves
axes[0].plot(hist_gd['loss'], label='Gradient Descent (lr=0.5)', linewidth=2)
axes[0].plot(hist_newton['loss'], 'o-', label="Newton's Method", linewidth=2, markersize=8)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Loss Over Training Steps')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: The unstable case
axes[1].plot(hist_gd['loss'][:20], label='GD (lr=0.5)', linewidth=2)
axes[1].plot(hist_gd_unstable['loss'], label='GD (lr=1.5) — BACKFIRE', linewidth=2, color='red')
axes[1].plot(hist_newton['loss'][:6], 'o-', label="Newton's Method", linewidth=2, markersize=8)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('MSE Loss')
axes[1].set_title('The Backfire: Learning Rate Too Large')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, max(hist_gd_unstable['loss'][:10]) * 1.1)

plt.tight_layout()
plt.show()

## Step 6: Looking Inside the Hessian

The Hessian is the key difference. Let's look at what it actually contains and what gradient descent is missing.

In [ ]:
print("The Hessian:")
print(f"  H = (2/n) * X^T X")
print(f"")
print(f"  H = [{H[0,0]:.4f}  {H[0,1]:.4f}]")
print(f"      [{H[1,0]:.4f}  {H[1,1]:.4f}]")
print(f"")
print(f"The diagonal entries ({H[0,0]:.4f} and {H[1,1]:.4f}) are the second derivatives")
print(f"of the loss with respect to each weight individually.")
print(f"")
print(f"The off-diagonal entry ({H[0,1]:.4f}) is the cross-term — how changing the")
print(f"intercept affects the optimal update for the slope, and vice versa.")
print(f"")
print(f"Gradient descent ignores this cross-term. Newton's method uses it.")
print(f"")
if abs(H[0,1]) > 0.01:
    print(f"The cross-term is {H[0,1]:.4f} — not zero. The parameters ARE entangled.")
    print(f"This is exactly the kind of interaction gradient descent misses.")
else:
    print(f"The cross-term is near zero — the parameters are nearly independent.")
    print(f"In this case, gradient descent doesn't miss much.")

## Step 7: Visualizing the Difference in Weight Space

Let's plot the loss surface and trace the paths that gradient descent and Newton's method take through weight space.

In [ ]:
# Create a grid of weight values around the optimum
w0_range = np.linspace(w_exact[0] - 50, w_exact[0] + 50, 100)
w1_range = np.linspace(w_exact[1] - 500, w_exact[1] + 500, 100)
W0, W1 = np.meshgrid(w0_range, w1_range)

# Compute loss at each point
Z = np.zeros_like(W0)
for i in range(W0.shape[0]):
    for j in range(W0.shape[1]):
        Z[i, j] = mse_loss(X, y, np.array([W0[i, j], W1[i, j]]))

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
contour = ax.contour(W0, W1, Z, levels=30, cmap='viridis', alpha=0.7)
ax.clabel(contour, inline=True, fontsize=8)

# Gradient descent path
gd_weights = np.array(hist_gd['weights'])
ax.plot(gd_weights[:, 0], gd_weights[:, 1], 'ro-', markersize=4, linewidth=1.5, label='Gradient Descent (50 steps)')

# Newton's path
newton_weights = np.array(hist_newton['weights'])
ax.plot(newton_weights[:, 0], newton_weights[:, 1], 'bs-', markersize=10, linewidth=2, label="Newton's Method (1 step)")

# Mark start and optimum
ax.plot(0, 0, 'k^', markersize=15, label='Start (0, 0)')
ax.plot(w_exact[0], w_exact[1], 'g*', markersize=20, label='Optimum')

ax.set_xlabel('w₀ (intercept)')
ax.set_ylabel('w₁ (slope)')
ax.set_title('Paths Through Weight Space')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 8: The Fitted Lines

Finally, let's see the regression lines both methods produce.

In [ ]:
x_line = np.linspace(X_raw.min(), X_raw.max(), 100)

plt.figure(figsize=(8, 5))
plt.scatter(X_raw, y, alpha=0.4, s=20, label='Data')
plt.plot(x_line, w_gd[0] + w_gd[1] * x_line, 'r-', linewidth=2, label=f'GD (50 steps): y = {w_gd[0]:.1f} + {w_gd[1]:.1f}x')
plt.plot(x_line, w_newton[0] + w_newton[1] * x_line, 'b--', linewidth=2, label=f"Newton (1 step): y = {w_newton[0]:.1f} + {w_newton[1]:.1f}x")
plt.xlabel('BMI (standardized)')
plt.ylabel('Disease Progression')
plt.title('Fitted Regression Lines')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nGD final loss (50 steps):     {hist_gd['loss'][-1]:.4f}")
print(f"Newton final loss (1 step):   {hist_newton['loss'][-1]:.4f}")
print(f"Loss difference:              {abs(hist_gd['loss'][-1] - hist_newton['loss'][-1]):.6f}")

## The Takeaway

Newton's method reaches the exact minimum in **1 step** because it accounts for cross-parameter interactions via the Hessian. Gradient descent takes many steps because each step ignores those interactions — it computes each `.grad` assuming other weights are constant.

But here, we only have **2 parameters**. The Hessian is a 2×2 matrix. Inverting it is trivial.

A model with $N$ parameters has an $N \times N$ Hessian. For a model with 1 million parameters, that's a matrix with **1 trillion entries**. Storing it requires $O(N^2)$ memory. Inverting it costs $O(N^3)$ computation.

Gradient descent is $O(N)$ per step. It's wrong — it ignores cross-parameter interactions — but it's cheap enough to run millions of times. Newton's method is correct but computationally impossible at scale.

**Gradient descent wins not because it's right, but because it's fast enough to iterate.**